# Jointure de `bureau_balance` avec `bureau`

## Objectif du notebook

L'objectif est d'enrichir chaque crédit externe de `bureau.csv` avec une synthèse de son historique mensuel provenant de `bureau_balance.csv`. La table mensuelle contient plusieurs lignes par `SK_ID_BUREAU` : elle doit donc être agrégée avant la jointure pour éviter de multiplier les lignes de `bureau`.

La granularité attendue à la fin reste **une ligne par crédit externe** (`SK_ID_BUREAU`). Ce notebook s'arrête volontairement à la création et à l'analyse de `bureau_enrichi`. La jointure avec les applications sera étudiée plus tard. Lors de cette future étape, `application_train` et `application_test` devront être traitées simultanément avec exactement les mêmes transformations afin de produire des variables compatibles pour l'entraînement et la prédiction.

Ordre du traitement :

1. charger et valider les deux tables ;
2. créer quatre caractéristiques synthétiques par `SK_ID_BUREAU` ;
3. effectuer une jointure gauche vers `bureau` ;
4. vérifier la granularité et analyser la table enrichie ;
5. exporter `bureau_enrichi` dans `data/processed/`.

In [2]:
from pathlib import Path

import pandas as pd

# ---------- Chemins des données ----------
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
BUREAU_PATH = RAW_DIR / "bureau.csv"
BUREAU_BALANCE_PATH = RAW_DIR / "bureau_balance.csv"
BUREAU_ENRICHI_PATH = PROCESSED_DIR / "bureau_enrichi.csv"

for data_path in [BUREAU_PATH, BUREAU_BALANCE_PATH]:
    if not data_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {data_path.resolve()}")

## Chargement des données

Des types numériques compacts sont utilisés pour `bureau_balance`, car cette table contient plus de 27 millions de lignes. Cela réduit la mémoire nécessaire sans modifier les valeurs.

In [3]:
# ---------- Chargement des données ----------
bureau = pd.read_csv(BUREAU_PATH)
bureau_balance = pd.read_csv(
    BUREAU_BALANCE_PATH,
    dtype={
        "SK_ID_BUREAU": "int32",
        "MONTHS_BALANCE": "int8",
        "STATUS": "category",
    },
)

print(f"bureau : {bureau.shape[0]:,} lignes et {bureau.shape[1]} colonnes")
print(
    f"bureau_balance : {bureau_balance.shape[0]:,} lignes "
    f"et {bureau_balance.shape[1]} colonnes"
)
display(bureau.head())
display(bureau_balance.head())

bureau : 1,716,428 lignes et 17 colonnes
bureau_balance : 27,299,925 lignes et 3 colonnes


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


## Validation des clés et des statuts

`SK_ID_BUREAU` doit être unique dans `bureau`. Dans `bureau_balance`, la combinaison de cet identifiant avec `MONTHS_BALANCE` doit être unique, puisqu'elle représente l'état d'un crédit pendant un mois donné. Les statuts attendus sont `0` à `5`, `C` (crédit clôturé) et `X` (statut inconnu).

In [4]:
# ---------- Validation des données ----------
BUREAU_ID = "SK_ID_BUREAU"
MONTH_COLUMN = "MONTHS_BALANCE"
STATUS_COLUMN = "STATUS"
EXPECTED_STATUS = {"0", "1", "2", "3", "4", "5", "C", "X"}

required_bureau_columns = {BUREAU_ID, "SK_ID_CURR"}
required_balance_columns = {BUREAU_ID, MONTH_COLUMN, STATUS_COLUMN}

if not required_bureau_columns.issubset(bureau.columns):
    missing = required_bureau_columns.difference(bureau.columns)
    raise KeyError(f"Colonnes absentes de bureau : {sorted(missing)}")
if not required_balance_columns.issubset(bureau_balance.columns):
    missing = required_balance_columns.difference(bureau_balance.columns)
    raise KeyError(f"Colonnes absentes de bureau_balance : {sorted(missing)}")

observed_status = set(bureau_balance[STATUS_COLUMN].dropna().astype(str).unique())
unexpected_status = observed_status.difference(EXPECTED_STATUS)

assert bureau[BUREAU_ID].notna().all(), "SK_ID_BUREAU contient des valeurs manquantes."
assert bureau[BUREAU_ID].is_unique, "SK_ID_BUREAU n'est pas unique dans bureau."
assert bureau_balance[list(required_balance_columns)].notna().all().all(), (
    "Une colonne obligatoire de bureau_balance contient des valeurs manquantes."
)
assert not bureau_balance.duplicated([BUREAU_ID, MONTH_COLUMN]).any(), (
    "Plusieurs lignes décrivent le même crédit pour le même mois."
)
assert not unexpected_status, f"Statuts inattendus : {sorted(unexpected_status)}"

print("Validation des clés et des statuts réussie.")
display(bureau_balance[STATUS_COLUMN].value_counts().sort_index().to_frame("effectif"))

Validation des clés et des statuts réussie.


,effectif
STATUS,
0,7499507
1,242347
2,23419
3,8924
4,5847
5,62406
C,13646993
X,5810482


## Création des caractéristiques de `bureau_balance`

Quatre caractéristiques complémentaires sont calculées pour chaque crédit :

- `BB_MONTH_COUNT` : nombre de mois d'historique disponibles ;
- `BB_DPD_RATIO` : proportion de mois comportant un retard (`STATUS` de `1` à `5`) ;
- `BB_EVER_SEVERE_DPD` : présence d'au moins un retard sévère de 61 jours ou plus (`STATUS` de `3` à `5`) ;
- `BB_RECENT_DPD_12M` : présence d'au moins un retard pendant les 12 mois les plus récents.

Les valeurs `C` et `X` ne sont pas assimilées à une absence de retard : elles sont exclues des indicateurs de retard.

In [5]:
# ---------- Feature engineering ----------
dpd_status = ["1", "2", "3", "4", "5"]
severe_dpd_status = ["3", "4", "5"]

bureau_balance = bureau_balance.assign(
    _IS_DPD=bureau_balance[STATUS_COLUMN].isin(dpd_status).astype("int8"),
    _IS_SEVERE_DPD=(
        bureau_balance[STATUS_COLUMN].isin(severe_dpd_status).astype("int8")
    ),
    _IS_RECENT_DPD=(
        bureau_balance[STATUS_COLUMN].isin(dpd_status)
        & bureau_balance[MONTH_COLUMN].ge(-12)
    ).astype("int8"),
)

bureau_balance_agg = (
    bureau_balance
    .groupby(BUREAU_ID, as_index=False, observed=True)
    .agg(
        BB_MONTH_COUNT=(MONTH_COLUMN, "size"),
        BB_DPD_RATIO=("_IS_DPD", "mean"),
        BB_EVER_SEVERE_DPD=("_IS_SEVERE_DPD", "max"),
        BB_RECENT_DPD_12M=("_IS_RECENT_DPD", "max"),
    )
)

bureau_balance_agg["BB_MONTH_COUNT"] = (
    bureau_balance_agg["BB_MONTH_COUNT"].astype("int16")
)
bureau_balance_agg["BB_DPD_RATIO"] = (
    bureau_balance_agg["BB_DPD_RATIO"].astype("float32")
)
bureau_balance_agg["BB_EVER_SEVERE_DPD"] = (
    bureau_balance_agg["BB_EVER_SEVERE_DPD"].astype("int8")
)
bureau_balance_agg["BB_RECENT_DPD_12M"] = (
    bureau_balance_agg["BB_RECENT_DPD_12M"].astype("int8")
)

assert bureau_balance_agg[BUREAU_ID].is_unique
del bureau_balance
print(f"Table agrégée : {bureau_balance_agg.shape[0]:,} crédits")
bureau_balance_agg.head()

Table agrégée : 817,395 crédits


,SK_ID_BUREAU,BB_MONTH_COUNT,BB_DPD_RATIO,BB_EVER_SEVERE_DPD,BB_RECENT_DPD_12M
0,5001709,97,0.0,0,0
1,5001710,83,0.0,0,0
2,5001711,4,0.0,0,0
3,5001712,19,0.0,0,0
4,5001713,22,0.0,0,0


## Jointure avec `bureau`

La jointure est effectuée sur `SK_ID_BUREAU`. Une jointure gauche conserve tous les crédits de `bureau`, y compris ceux qui ne possèdent aucun historique mensuel. `validate="one_to_one"` protège la granularité en déclenchant une erreur si l'une des tables contient plusieurs lignes pour un même crédit.

In [6]:
# ---------- Analyse de la couverture des clés ----------
bureau_ids = pd.Index(bureau[BUREAU_ID])
balance_ids = pd.Index(bureau_balance_agg[BUREAU_ID])
orphan_balance_ids = balance_ids.difference(bureau_ids)
bureau_without_balance_ids = bureau_ids.difference(balance_ids)

print(f"Identifiants de bureau_balance absents de bureau : {len(orphan_balance_ids):,}")
print(f"Crédits de bureau sans historique mensuel : {len(bureau_without_balance_ids):,}")

Identifiants de bureau_balance absents de bureau : 43,041
Crédits de bureau sans historique mensuel : 942,074


In [7]:
# ---------- Jointure des données ----------
bureau_enrichi = bureau.merge(
    bureau_balance_agg,
    on=BUREAU_ID,
    how="left",
    validate="one_to_one",
)

bureau_enrichi["BB_HAS_HISTORY"] = (
    bureau_enrichi["BB_MONTH_COUNT"].notna().astype("int8")
)
bureau_enrichi["BB_MONTH_COUNT"] = (
    bureau_enrichi["BB_MONTH_COUNT"].fillna(0).astype("int16")
)

print(f"bureau enrichi : {bureau_enrichi.shape[0]:,} lignes et {bureau_enrichi.shape[1]} colonnes")
bureau_enrichi.head()

bureau enrichi : 1,716,428 lignes et 22 colonnes


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,BB_MONTH_COUNT,BB_DPD_RATIO,BB_EVER_SEVERE_DPD,BB_RECENT_DPD_12M,BB_HAS_HISTORY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,NaN,0.0,Consumer credit,-131,NaN,0,NaN,NaN,NaN,0
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,NaN,0.0,Credit card,-20,NaN,0,NaN,NaN,NaN,0
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,NaN,0.0,Consumer credit,-16,NaN,0,NaN,NaN,NaN,0
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,NaN,0.0,Credit card,-16,NaN,0,NaN,NaN,NaN,0
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,NaN,0.0,Consumer credit,-21,NaN,0,NaN,NaN,NaN,0


## Analyse et validation de `bureau_enrichi`

Les contrôles suivants vérifient que la jointure n'a supprimé ni multiplié aucun crédit. Les valeurs manquantes des indicateurs de retard sont conservées lorsqu'il n'existe aucun historique : `BB_HAS_HISTORY` permet de distinguer cette situation d'un historique observé sans retard.

In [8]:
# ---------- Validation de la table enrichie ----------
assert len(bureau_enrichi) == len(bureau), (
    "La jointure a modifié le nombre de lignes de bureau."
)
assert bureau_enrichi[BUREAU_ID].is_unique, (
    "SK_ID_BUREAU n'est plus unique après la jointure."
)
assert bureau_enrichi["BB_HAS_HISTORY"].eq(
    bureau_enrichi["BB_DPD_RATIO"].notna()
).all(), "L'indicateur de présence d'historique est incohérent."

summary = pd.DataFrame(
    {
        "indicateur": [
            "Nombre de crédits",
            "Crédits avec historique",
            "Crédits sans historique",
            "Crédits avec retard sévère",
            "Crédits avec retard récent",
        ],
        "valeur": [
            len(bureau_enrichi),
            int(bureau_enrichi["BB_HAS_HISTORY"].sum()),
            int(bureau_enrichi["BB_HAS_HISTORY"].eq(0).sum()),
            int(bureau_enrichi["BB_EVER_SEVERE_DPD"].eq(1).sum()),
            int(bureau_enrichi["BB_RECENT_DPD_12M"].eq(1).sum()),
        ],
    }
)
display(summary)
display(
    bureau_enrichi[
        [
            "BB_MONTH_COUNT",
            "BB_DPD_RATIO",
            "BB_EVER_SEVERE_DPD",
            "BB_RECENT_DPD_12M",
            "BB_HAS_HISTORY",
        ]
    ].describe()
)
print("Toutes les validations sont réussies.")

,indicateur,valeur
0,Nombre de crédits,1716428
1,Crédits avec historique,774354
2,Crédits sans historique,942074
3,Crédits avec retard sévère,7300
4,Crédits avec retard récent,34072


,BB_MONTH_COUNT,BB_DPD_RATIO,BB_EVER_SEVERE_DPD,BB_RECENT_DPD_12M,BB_HAS_HISTORY
count,1.716428e+06,774354.000000,774354.000000,774354.000000,1.716428e+06
mean,1.408724e+01,0.013974,0.009427,0.044001,4.511427e-01
std,2.214124e+01,0.059376,0.096635,0.205096,4.976074e-01
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
25%,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
50%,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00
75%,2.200000e+01,0.000000,0.000000,0.000000,1.000000e+00
max,9.700000e+01,1.000000,1.000000,1.000000,1.000000e+00


Toutes les validations sont réussies.


## Export de la table enrichie

La table est enregistrée dans `data/processed/`. Elle conserve une ligne par crédit externe et servira de point de départ à la future réflexion sur l'agrégation par client et la jointure avec les applications.

In [9]:
# ---------- Export des données ----------
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
bureau_enrichi.to_csv(BUREAU_ENRICHI_PATH, index=False)

print(f"Table exportée vers : {BUREAU_ENRICHI_PATH.resolve()}")

Table exportée vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\bureau_enrichi.csv


# Préparation de la jointure des tables `bureau_enrichi` avec `application`

`bureau_enrichi` contient encore plusieurs crédits pour un même client. Une agrégation par `SK_ID_CURR` est donc obligatoire avant toute jointure afin de conserver une ligne par client. Les applications d'entraînement et de test restent dans deux tables séparées, mais reçoivent exactement les mêmes variables.

### Colonnes transformées

- `SK_ID_BUREAU` est compté pour mesurer le nombre de crédits.
- `CREDIT_ACTIVE` devient un nombre et une proportion de crédits actifs.
- `CREDIT_CURRENCY` et `CREDIT_TYPE` deviennent des nombres de crédits par catégorie.
- `DAYS_CREDIT` fournit la date relative du crédit le plus récent.
- Les montants de crédit, dette et impayé sont additionnés par client.
- `CREDIT_DAY_OVERDUE` fournit le nombre de crédits en retard et le retard maximal.
- `CNT_CREDIT_PROLONG` fournit le total et le nombre moyen de prolongations par crédit.
- Les indicateurs issus de `bureau_balance` résument les retards sévères, récents et la couverture de l'historique mensuel.

### Colonnes non retenues pour cette première version

- `DAYS_CREDIT_ENDDATE`, `DAYS_ENDDATE_FACT` et `DAYS_CREDIT_UPDATE` : informations temporelles secondaires qui demanderaient une analyse dédiée.
- `AMT_CREDIT_MAX_OVERDUE`, `AMT_CREDIT_SUM_LIMIT` et `AMT_ANNUITY` : montants complémentaires, potentiellement redondants ou très incomplets.
- `BB_MONTH_COUNT` et `BB_DPD_RATIO` : détails au niveau du crédit non retenus directement ; la couverture et les indicateurs de retard synthétiques sont conservés.

Ces colonnes restent disponibles dans `bureau_enrichi.csv` et pourront être réintroduites ultérieurement si leur intérêt est démontré.

In [10]:
# ---------- Variables intermédiaires au niveau du crédit ----------
CLIENT_ID = "SK_ID_CURR"

bureau_enrichi = bureau_enrichi.assign(
    _IS_ACTIVE=bureau_enrichi["CREDIT_ACTIVE"].eq("Active").astype("int8"),
    _IS_OVERDUE=bureau_enrichi["CREDIT_DAY_OVERDUE"].gt(0).astype("int8"),
)

bureau_grouped = bureau_enrichi.groupby(CLIENT_ID, observed=True)

In [11]:
# ---------- Agrégations principales par client ----------
bureau_par_client = bureau_grouped.agg(
    BUREAU_CREDIT_COUNT=(BUREAU_ID, "size"),
    BUREAU_ACTIVE_CREDIT_COUNT=("_IS_ACTIVE", "sum"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_OVERDUE_CREDIT_COUNT=("_IS_OVERDUE", "sum"),
    BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_PROLONGATION_SUM=("CNT_CREDIT_PROLONG", "sum"),
    BUREAU_EVER_SEVERE_DPD=("BB_EVER_SEVERE_DPD", "max"),
    BUREAU_EVER_RECENT_DPD_12M=("BB_RECENT_DPD_12M", "max"),
    BUREAU_BB_HISTORY_CREDIT_COUNT=("BB_HAS_HISTORY", "sum"),
)

amount_columns = {
    "AMT_CREDIT_SUM": "BUREAU_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT": "BUREAU_DEBT_SUM",
    "AMT_CREDIT_SUM_OVERDUE": "BUREAU_OVERDUE_SUM",
}
amounts_by_client = (
    bureau_grouped[list(amount_columns)]
    .sum(min_count=1)
    .rename(columns=amount_columns)
)
bureau_par_client = bureau_par_client.join(amounts_by_client)

In [12]:
# ---------- Comptage des catégories par client ----------
def normalize_category(value):
    return (
        str(value).upper()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )


def count_categories_by_client(data, column, prefix):
    counts = pd.crosstab(data[CLIENT_ID], data[column])
    counts.columns = [
        f"{prefix}_{normalize_category(category)}_COUNT"
        for category in counts.columns
    ]
    if counts.columns.duplicated().any():
        raise ValueError(f"Noms de colonnes dupliqués après encodage de {column}.")
    return counts


currency_counts = count_categories_by_client(
    bureau_enrichi, "CREDIT_CURRENCY", "BUREAU_CURRENCY"
)
credit_type_counts = count_categories_by_client(
    bureau_enrichi, "CREDIT_TYPE", "BUREAU_CREDIT_TYPE"
)
bureau_par_client = bureau_par_client.join(currency_counts).join(credit_type_counts)

In [13]:
# ---------- Ratios au niveau du client ----------
bureau_par_client["BUREAU_ACTIVE_CREDIT_RATIO"] = (
    bureau_par_client["BUREAU_ACTIVE_CREDIT_COUNT"]
    / bureau_par_client["BUREAU_CREDIT_COUNT"]
)
bureau_par_client["BUREAU_PROLONGATION_PER_CREDIT"] = (
    bureau_par_client["BUREAU_PROLONGATION_SUM"]
    / bureau_par_client["BUREAU_CREDIT_COUNT"]
)
bureau_par_client["BUREAU_BB_HISTORY_RATIO"] = (
    bureau_par_client["BUREAU_BB_HISTORY_CREDIT_COUNT"]
    / bureau_par_client["BUREAU_CREDIT_COUNT"]
)
credit_denominator = bureau_par_client["BUREAU_CREDIT_SUM"].mask(
    bureau_par_client["BUREAU_CREDIT_SUM"].eq(0)
)
bureau_par_client["BUREAU_DEBT_TO_CREDIT_RATIO"] = (
    bureau_par_client["BUREAU_DEBT_SUM"] / credit_denominator
)

bureau_par_client = bureau_par_client.reset_index()
assert bureau_par_client[CLIENT_ID].is_unique
print(
    f"bureau_par_client : {bureau_par_client.shape[0]:,} lignes "
    f"et {bureau_par_client.shape[1]} colonnes"
)
bureau_par_client.head()

bureau_par_client : 305,811 lignes et 36 colonnes


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_ACTIVE_CREDIT_COUNT,BUREAU_DAYS_CREDIT_MAX,BUREAU_OVERDUE_CREDIT_COUNT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_PROLONGATION_SUM,BUREAU_EVER_SEVERE_DPD,BUREAU_EVER_RECENT_DPD_12M,BUREAU_BB_HISTORY_CREDIT_COUNT,...,BUREAU_CREDIT_TYPE_LOAN_FOR_WORKING_CAPITAL_REPLENISHMENT_COUNT,BUREAU_CREDIT_TYPE_MICROLOAN_COUNT,BUREAU_CREDIT_TYPE_MOBILE_OPERATOR_LOAN_COUNT,BUREAU_CREDIT_TYPE_MORTGAGE_COUNT,BUREAU_CREDIT_TYPE_REAL_ESTATE_LOAN_COUNT,BUREAU_CREDIT_TYPE_UNKNOWN_TYPE_OF_LOAN_COUNT,BUREAU_ACTIVE_CREDIT_RATIO,BUREAU_PROLONGATION_PER_CREDIT,BUREAU_BB_HISTORY_RATIO,BUREAU_DEBT_TO_CREDIT_RATIO
0,100001,7,3,-49,0,0,0,0.0,1.0,7,...,0,0,0,0,0,0,0.428571,0.0,1.0,0.410555
1,100002,8,2,-103,0,0,0,0.0,0.0,8,...,0,0,0,0,0,0,0.250000,0.0,1.0,0.284122
2,100003,4,1,-606,0,0,0,NaN,NaN,0,...,0,0,0,0,0,0,0.250000,0.0,0.0,0.000000
3,100004,2,0,-408,0,0,0,NaN,NaN,0,...,0,0,0,0,0,0,0.000000,0.0,0.0,0.000000
4,100005,3,2,-62,0,0,0,0.0,0.0,3,...,0,0,0,0,0,0,0.666667,0.0,1.0,0.864992


## Jointure séparée avec le train et le test

Les deux applications préparées sont chargées séparément. Les jointures gauches conservent tous les clients, y compris ceux absents de `bureau`. `BUREAU_HAS_HISTORY` distingue alors l'absence d'un dossier Bureau d'un dossier observé sans retard.

In [14]:
# ---------- Chargement des applications préparées ----------
TRAIN_APPLICATION_PATH = PROCESSED_DIR / "application_train_clean_encoded.csv"
TEST_APPLICATION_PATH = PROCESSED_DIR / "application_test_clean_encoded.csv"
TRAIN_FINAL_PATH = PROCESSED_DIR / "application_train_enrichi_bureau_encoded.csv"
TEST_FINAL_PATH = PROCESSED_DIR / "application_test_enrichi_bureau_encoded.csv"

for data_path in [TRAIN_APPLICATION_PATH, TEST_APPLICATION_PATH]:
    if not data_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {data_path.resolve()}")

application_train = pd.read_csv(TRAIN_APPLICATION_PATH)
application_test = pd.read_csv(TEST_APPLICATION_PATH)

assert application_train[CLIENT_ID].is_unique
assert application_test[CLIENT_ID].is_unique

In [15]:
# ---------- Jointures avec les applications ----------
application_train_bureau = application_train.merge(
    bureau_par_client, on=CLIENT_ID, how="left", validate="one_to_one"
)
application_test_bureau = application_test.merge(
    bureau_par_client, on=CLIENT_ID, how="left", validate="one_to_one"
)

train_history_indicator = (
    application_train_bureau["BUREAU_CREDIT_COUNT"]
    .notna()
    .astype("int8")
    .rename("BUREAU_HAS_HISTORY")
)
test_history_indicator = (
    application_test_bureau["BUREAU_CREDIT_COUNT"]
    .notna()
    .astype("int8")
    .rename("BUREAU_HAS_HISTORY")
)
application_train_bureau = pd.concat(
    [application_train_bureau, train_history_indicator], axis=1
)
application_test_bureau = pd.concat(
    [application_test_bureau, test_history_indicator], axis=1
)

In [16]:
# ---------- Validation des jeux finaux ----------
assert len(application_train_bureau) == len(application_train)
assert len(application_test_bureau) == len(application_test)
assert application_train_bureau[CLIENT_ID].is_unique
assert application_test_bureau[CLIENT_ID].is_unique
assert "TARGET" in application_train_bureau.columns
assert "TARGET" not in application_test_bureau.columns

train_features = application_train_bureau.drop(columns="TARGET").columns.tolist()
assert train_features == application_test_bureau.columns.tolist(), (
    "Les variables du train et du test ne sont pas alignées."
)

display(
    pd.DataFrame(
        {
            "jeu de données": ["train", "test"],
            "nombre de lignes": [len(application_train_bureau), len(application_test_bureau)],
            "nombre de colonnes": [
                application_train_bureau.shape[1],
                application_test_bureau.shape[1],
            ],
            "clients avec historique Bureau": [
                int(application_train_bureau["BUREAU_HAS_HISTORY"].sum()),
                int(application_test_bureau["BUREAU_HAS_HISTORY"].sum()),
            ],
        }
    )
)
print("Toutes les validations des applications enrichies sont réussies.")

,jeu de données,nombre de lignes,nombre de colonnes,clients avec historique Bureau
0,train,307510,294,263490
1,test,48744,293,42320


Toutes les validations des applications enrichies sont réussies.


In [17]:
# ---------- Export des applications enrichies ----------
application_train_bureau.to_csv(TRAIN_FINAL_PATH, index=False)
application_test_bureau.to_csv(TEST_FINAL_PATH, index=False)

print(f"Train exporté vers : {TRAIN_FINAL_PATH.resolve()}")
print(f"Test exporté vers : {TEST_FINAL_PATH.resolve()}")

Train exporté vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\application_train_enrichi_bureau_encoded.csv
Test exporté vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\application_test_enrichi_bureau_encoded.csv


## Conclusion

`bureau_balance` a d'abord été agrégé par crédit et joint à `bureau`. `bureau_enrichi` a ensuite été agrégé par client avant deux jointures séparées et identiques avec les applications d'entraînement et de test. Les deux jeux finaux conservent une ligne par `SK_ID_CURR` et possèdent les mêmes variables explicatives.